# Génération de règles YARA par RAG

Un analyste décrit un comportement malveillant ; le système retrouve des règles proches dans une base
de ~11 000 règles YARA publiques, puis un LLM écrit une nouvelle règle **vérifiée par le compilateur YARA**.

| Étape | Méthode |
|---|---|
| Recherche | BM25 (mots-clés + chaînes des règles) et embeddings `bge-base-en-v1.5` + FAISS, fusion RRF |
| Re-classement | cross-encoder `ms-marco-MiniLM-L-6-v2` |
| Génération | TinyLlama 1.1B, Phi-2 2.7B ou Mistral-7B Instruct (4 bits) |
| Contrôle | réparation ciblée + compilation `yara-python` ; l'agent renvoie l'erreur au LLM |

**Exécution : Colab avec GPU T4** (Exécution → Modifier le type d'exécution → GPU). Tout le code est dans le
paquet `yara_rag/` du dépôt ; ce notebook l'utilise.

## 1. Installation

In [ ]:
import os, sys, subprocess
REPO = "https://github.com/Minyurano/mini-projet-NLP.git"
if not os.path.exists("mini-projet-NLP") and not os.path.exists("yara_rag"):
    subprocess.run(["git", "clone", "--quiet", REPO], check=True)
os.chdir("mini-projet-NLP") if os.path.exists("mini-projet-NLP") else None
%pip install -q -r requirements.txt -r requirements-llm.txt
sys.path.insert(0, os.getcwd())
import torch
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "aucun (génération très lente)")

## 2. Base de règles
Reconstruite depuis des dépôts publics (Yara-Rules, signature-base de Florian Roth, ReversingLabs). Seules les règles qui compilent sont gardées, les doublons sont retirés. Pour utiliser l'ancien fichier `yara_step2.csv`, mettez son chemin dans `DATA`.

In [ ]:
import subprocess, sys
from pathlib import Path
DATA = Path("data/yara_rules.csv")
if not DATA.exists():
    subprocess.run([sys.executable, "scripts/build_dataset.py", "--out", str(DATA)], check=True)

from yara_rag.dataset import load_dataset
df = load_dataset(DATA)
print(f"{len(df):,} règles")
df["type"].value_counts().head(12)

## 3. Index de recherche (BM25 + embeddings + re-classement)

In [ ]:
from sentence_transformers import CrossEncoder, SentenceTransformer
from yara_rag.retrieval import Retriever

embedder = SentenceTransformer("BAAI/bge-base-en-v1.5")
encode = lambda texts: embedder.encode(list(texts), batch_size=64, convert_to_numpy=True,
                                       normalize_embeddings=True, show_progress_bar=len(texts) > 100)
retriever = Retriever(df, encoder=encode, reranker=CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2"))

for h in retriever.docs(retriever.search_rerank("keylogger capturing keystrokes with GetAsyncKeyState", k=3)):
    print(f"[{h['type']}] {h['rule_name']}  —  {h['description'][:70]}")

### Qualité de la recherche
Part des 5 règles retrouvées qui ont le type attendu : 15 requêtes étiquetées, et 500 requêtes « leave-one-out » (la description d'une règle sert de requête, la règle elle-même est exclue).

In [ ]:
from yara_rag.metrics import retrieval_benchmark
retrieval_benchmark(retriever, df, methods={
    "BM25": retriever.search_bm25,
    "Hybride (RRF)": retriever.search_hybrid,
    "Hybride + rerank": retriever.search_rerank,
})

## 4. Modèle de langage
Un seul modèle en mémoire à la fois. Mistral-7B donne les meilleures règles mais demande ~5 Go de GPU en 4 bits.

In [ ]:
from yara_rag.generation import MODELS, HFGenerator
LLM = "TinyLlama (1.1B)"          # ou "Phi-2 (2.7B)", "Mistral-7B Instruct"
generator = HFGenerator(LLM)
print("Modèle chargé :", generator.name)

## 5. Essai des trois architectures

In [ ]:
from yara_rag.pipeline import YaraRAG
rag = YaraRAG(retriever, generator)
query = "Ransomware that encrypts files with AES, deletes shadow copies and asks for a Bitcoin ransom"
for arch in YaraRAG.ARCHITECTURES:
    g = rag.run(query, arch)
    print(f"=== {arch} — {'compile' if g.valid else 'ERREUR : ' + str(g.error)} — {g.latency:.1f} s — {g.attempts} essai(s)")
    if g.fixes: print("corrections :", "; ".join(g.fixes))
    print(g.rule, "\n")

## 6. Évaluation
15 requêtes étiquetées × 3 architectures. Mesures : règle compilée (`syntax_valid`), précision du type des règles retrouvées, couverture des mots-clés techniques de la requête, ancrage des chaînes dans les règles retrouvées et copie pure d'une règle existante, nombre d'essais, latence.

In [ ]:
from yara_rag.metrics import evaluate, summary
results = evaluate(rag)
Path("results").mkdir(exist_ok=True)
results.to_csv(f"results/eval_{LLM.split()[0].lower()}.csv", index=False)
summary(results)

## 7. Interface

In [ ]:
from yara_rag.app import ModelCache, build_app
demo = build_app(retriever, ModelCache(generator), default_model=LLM)
demo.launch(share=True)